Tutorial stuff to get PyPSA working in a single-node model, building towards the PyPSA-RSA style model

### Import cell

In [ ]:
import pypsa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

### Explore results

In [ ]:
results_path = Path("~/showcase/pypsa-rsa/results/TEST/network/capacity-TEST.nc").expanduser()
input_path = Path("~/showcase/pypsa-rsa/networks/TEST/elec/capacity-TEST.nc").expanduser()
dispatch_path = Path("~/showcase/pypsa-rsa/networks/TEST/elec/dispatch-2030.nc").expanduser()

In [ ]:
n_0 = pypsa.Network(input_path)
n_1 = pypsa.Network(dispatch_path)

In [ ]:
n.statistics.installed_capacity().reset_index().where(lambda x: (x['component'] == 'Generator') | (x['component'] == 'StorageUnit'))\
    .groupby('carrier').sum().drop(columns=['component']).rename(columns={2030: 'capacity [MW]'}).plot.bar()
plt.ylabel('capacity [MW]')
plt.title('Installed Capacity by Carrier')

In [ ]:
n.statistics.supply(comps=['Generator', 'StorageUnit'], aggregate_time=False)[2030].stack().unstack('carrier').reset_index().drop(columns=['component']).groupby('timestep').sum().plot(kind='are', stacked=True)

In [ ]:
n.carriers

In [ ]:
n.storage_units

In [ ]:
def plot_capacity_expansion(n):
    """
    Plot the capacity expansion results for generators and storage units.
    
    Args:
        n (pypsa.Network): The PyPSA network object containing the results.
    """
    gen_p_exp = n.generators.p_nom_opt - n.generators.p_nom
    gen_p_exp = gen_p_exp.groupby([n.generators.carrier, n.generators.bus]).sum()

    storage_p_exp = n.storage_units.p_nom_opt - n.storage_units.p_nom
    storage_p_exp = storage_p_exp.groupby([n.storage_units.carrier, n.storage_units.bus]).sum()

    gen_p_exp = pd.concat([gen_p_exp, storage_p_exp], axis=0)

    gen_p_exp.unstack('bus').plot(
        kind='bar', 
        stacked=True, 
        title='Newly built capacity per carrier and bus',
        ylabel='Capacity [MW]',
        xlabel='Carrier'
    )

In [ ]:
import cartopy.crs as ccrs
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw={'projection': ccrs.EqualEarth()})

n.plot(
    ax=ax,
    margin=0.5,
    line_flow="mean",
    link_flow="mean",
    line_widths=2.7,
    link_widths=0.2,
    projection=ccrs.EqualEarth(),
    color_geomap=True
)
plt.show()

In [ ]:
n.storage_units_t.p_dispatch.sum(axis=1).plot()